- Update des prix et des stocks de la library 
- Check de la BOM avec la Library
- Check si les composant sont disponible et pricing

In [1]:
CHEMIN_LIB = "lib/Component_library.xlsx"
CHEMIN_BOM = "data/Carte_radar_V0.2--BoM.xlsx"
#CHEMIN_BOM= "data/PAMI_MOTORISATION_V--BoM.xlsx"
import pandas as pd
from backend.BOM_function import *
from backend.find_component import search_in_lib,find_by_value
from backend.Pricing_lcsc import Update_Price_Stock

In [2]:
bom = pd.read_excel(CHEMIN_BOM, header=7).dropna(how="all").drop(columns=['Row'])
bom["LCSC_part_number"] = bom["LCSC_part_number"].astype("object")
bom["Comments"] = bom["Comments"].astype("object")


In [3]:
lib = pd.read_excel(CHEMIN_LIB)

for index, row in bom.iterrows():

    result = search_in_lib(lib, row)

    ref = row.get("Description", "")
    kk = row.get("References", "")
    value = row.get("Value", "")
    footprint = row.get("Footprint", "")

    if result is None or result.empty:
        print(f"[{index:>3}] {ref!r:25} {value!r:15} {footprint!r:20} -> AUCUN RESULTAT")
        continue

    nb = len(result)
    if nb == 1:
        lcsc = result.iloc[0]["description"]
        print(f"[{index:>3}] {ref!r:25} {kk} {value!r:15} {footprint!r:20} -> OK : {lcsc}")
    else:
        candidats = ", ".join(str(x) for x in result["Manufacturer Ref"].tolist())
        print(f"[{index:>3}] {ref!r:25} {kk} {value!r:15} {footprint!r:20} -> {nb} CANDIDATS : {candidats}")

[  0] 'Unpolarized capacitor, small symbol' C11 C16 '12pF'          ''                   -> OK : CAP CER 12pF 50V NP0 0402
[  1] 'Unpolarized capacitor, small symbol' C1 C3 C4 C5 C6 C7 C8 C9 C10 C12 C14 C19 C20 '100nF'         ''                   -> OK : CAP CER 100nF 10V X5R 0402
[  2] 'Capacitor 0603 1uF 50V X7R 10%' C15 '1uF'           ''                   -> OK : CAP CER 1uF 50V X7R 0603
[  3] 'Tantalum capacitor 0805 16V 4.7uF 10% E SR= 5 Ohm' C13 '4.7uF'         ''                   -> OK : 4.7uF ±10% 50V Ceramic Capacitor X7R 1206
[  4] 'Polarized capacitor'     C2 '4.7uF'         ''                   -> OK : CAP CER 4.7uF 6.3V X5R 0402
[  5] nan                       D1 D2 D9 'LED_Green'     ''                   -> OK : LED GREEN 0603 SMD
[  6] 'RGB LED with integrated controller' D10 D11 D12 D13 D14 D15 D16 D17 'WS2812B'       ''                   -> OK : Red, green, blue 5mm x 5mm square LED SMD5050-4P LED Indication - Discrete 
[  7] '74AHCT Single Tri-state Bus Buffer SOT2

In [14]:
Update_Price_Stock(CHEMIN_LIB)

SN74AHCT1G125DBVR C7484
LCSC - HTTP : 200
PTS810SJG250SMTRLFS C221895
LCSC - HTTP : 200
06035C105KAT2A C2182269
LCSC - HTTP : 200
CC0402JRNPO9BN120 C106201
LCSC - HTTP : 200
CC0402KRX5R5BB475 C541466
LCSC - HTTP : 200
CC0402KRX5R6BB104 C129131
LCSC - HTTP : 200
CL10A226MP8NUNE C86295
LCSC - HTTP : 200
CL31B475KBHNNNE C51205
LCSC - HTTP : 200
HGC0603R5106M350NTHJ C22367827
LCSC - HTTP : 200
PZ254R-12-10P C492433
LCSC - HTTP : 200
TCC0402C0G470J500AT C466230
LCSC - HTTP : 200
B4B-ZR(LF)(SN) C157997
LCSC - HTTP : 200
B6B-ZR-3.4(LF)(SN) C495631
LCSC - HTTP : 200
HC-1.25-3PLT C2845389
LCSC - HTTP : 200
HC-1.25-5PLT C2845391
LCSC - HTTP : 200
PM254-2-05-S-8.5 C3975150
LCSC - HTTP : 200
X32258MOB4SI C2682775
LCSC - HTTP : 200
ICM-42688-P C54308212
LCSC - HTTP : 200
HTMD-4020-6R8-M C54321652
LCSC - HTTP : 200
LDL1117S33R C435835
LCSC - HTTP : 200
HSMF-C165 C188726
LCSC - HTTP : 200
HSMG-C190 C188729
LCSC - HTTP : 200
WS2812B-V6 C52917433
LCSC - HTTP : 200
TB67H450FNG_EL C545417
LCSC - HTTP : 2

In [6]:
from backend.BOM_function import *
from backend.BOM_report import *
from backend.find_component import search_in_lib

for index, row in bom.iterrows():
    result = search_in_lib(lib, row)
    row = fill_bom_result(row, result)
    row = check_bom(row, result)
    bom.loc[index] = row

bom


,Description,Value,References,Quantity Per PCB,Manufacturer Ref,Manufacturer,LCSC_part_number,Comments
0,"Unpolarized capacitor, small symbol",12pF,C11 C16,2,CC0402JRNPO9BN120,Yageo,C106201,NaN
1,"Unpolarized capacitor, small symbol",100nF,C1 C3 C4 C5 C6 C7 C8 C9 C10 C12 C14 C19 C20,13,CC0402KRX5R6BB104,YAGEO,C129131,NaN
2,Capacitor 0603 1uF 50V X7R 10%,1uF,C15,1,06035C105KAT2A,KYOCERA AVX,C2182269,NaN
3,Tantalum capacitor 0805 16V 4.7uF 10% E SR= 5 Ohm,4.7uF,C13,1,TAJA475K016RNJ,KYOCERA AVX,C7187,REFERENCE MISMATCH | REFERENCE MISMATCH
4,Polarized capacitor,4.7uF,C2,1,CC0402KRX5R5BB475,YAGEO,C541466,NaN
5,NaN,LED_Green,D1 D2 D9,3,HSMG-C190,BROADCOM,C188729,NaN
6,RGB LED with integrated controller,WS2812B,D10 D11 D12 D13 D14 D15 D16 D17,8,WS2812B-B/W,Worldsemi,C114586,REFERENCE MISMATCH | REFERENCE MISMATCH
7,74AHCT Single Tri-state Bus Buffer SOT23 SN74A...,SN74AHCT1G125DBVR,IC1,1,SN74AHCT1G125DBVR,Texas Instruments,C7484,NaN
8,"Generic connectable mounting pin connector, si...",Conn_01x03_MountingPin,J12 J15,2,HC-1.25-3PLT,HCTL,C2845389,NaN
9,"Generic connectable mounting pin connector, si...",Conn_01x05_MountingPin,J11,1,HC-1.25-5PLT,HCTL,C2845391,NaN


In [8]:
report = build_bom_report(bom, lib, 5, quantity_column=None)
print_report(report,5)

RAPPORT BOM - pour 5 PCB
Nombre de lignes BOM       : 17
Lignes OK                  : 13
Lignes incomplètes/erreurs : 4
Prix total estimé          : 16.63 EUR
------------------------------------------------------------
Composants à vérifier :
        Value Manufacturer Ref LCSC_part_number             Status                                                                                                            Comments
        4.7uF   TAJA475K016RNJ            C7187 REFERENCE MISMATCH                                                                             REFERENCE MISMATCH | REFERENCE MISMATCH
      WS2812B      WS2812B-B/W          C114586 REFERENCE MISMATCH                                                                             REFERENCE MISMATCH | REFERENCE MISMATCH
  LDL112PV33R      LDL112PV33R         C2971424 REFERENCE MISMATCH                                                                             REFERENCE MISMATCH | REFERENCE MISMATCH
STM32G070CBTx    STM32G0